# 詩織 LoRA v4 ブラッシュアップ推論ノートブック（Kaggle）

学習済みLoRA（train-shiori-lora-kaggle-v4 Version 2 の出力 shiori_oneesan_v4.safetensors）を **Add Input** で接続して使う。学習は行わない。
v4通常推論（28ステップ）との違いは「推論側の仕上げ」だけ：
- ステップ数を 28 → 40 に増やす
- 生成後に 1.5倍アップスケール＋img2img（strength 0.35）で描き直しパスを入れて、顔・髪の質感を整える
- プロンプトとシード（42）は v4 テストと同一にして、仕上げの効果だけを比較できるようにする
生成は2枚：①標準プロンプト（商店街カフェ）、②髪チェック用（肩に流した髪のアップ）。

In [ ]:
# GPU確認
!nvidia-smi


In [ ]:
# 依存関係の用意（torchao は古い版が入っていると peft が ImportError になるため外す）
!pip install -q diffusers transformers accelerate peft huggingface_hub
!pip uninstall -y torchao

# ベースモデル（Animagine XL 4.0）を取得
from huggingface_hub import hf_hub_download
import shutil, os
p = hf_hub_download(repo_id='cagliostrolab/animagine-xl-4.0', filename='animagine-xl-4.0.safetensors')
shutil.copy(p, '/kaggle/working/animagine-xl-4.0.safetensors')
print('model ready:', os.path.getsize('/kaggle/working/animagine-xl-4.0.safetensors'))


In [ ]:
# LoRA を読み込んでブラッシュアップ生成（2枚）
import glob, os, gc, torch
from diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
from PIL import Image
cands = [c for c in glob.glob('/kaggle/input/**/shiori_oneesan_v4.safetensors', recursive=True)]
print('lora candidates:', cands)
assert cands, 'shiori_oneesan_v4.safetensors not found in /kaggle/input'
lora_path = cands[0]
lora_dir = os.path.dirname(lora_path)
print('lora size:', os.path.getsize(lora_path))
neg = 'lowres, bad anatomy, bad hands, extra fingers, worst quality, blurry, multiple girls, 2girls, extra person, tsurime, sharp eyes, narrow eyes'
os.makedirs('/kaggle/working/output', exist_ok=True)
prompts = [
    ('test_outing_v4b.png', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, pink cardigan, shopping street, cafe terrace, sunny day'),
    ('test_hair_v4b.png', 'shiori_oneesan, 1woman, solo, very long hair, straight black hair, matte black hair, hime cut, blunt bangs, side part, purple eyes, gentle eyes, droopy eyes, tareme, gentle smile, soft expression, hair over shoulder, white blouse, indoors, window, daylight, close-up portrait'),
]

# 第1パス: テキストから生成（40ステップ）
pipe = StableDiffusionXLPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe.load_lora_weights(lora_dir, weight_name='shiori_oneesan_v4.safetensors')
bases = []
for name, prompt in prompts:
    g = torch.Generator(device='cuda').manual_seed(42)
    img = pipe(prompt, negative_prompt=neg, width=1024, height=1024, num_inference_steps=40, guidance_scale=7, generator=g).images[0]
    bases.append((name, prompt, img))
    print('base done:', name)
del pipe
gc.collect()
torch.cuda.empty_cache()

# 第2パス: 1.5倍に拡大して img2img で仕上げ（strength 0.35）
pipe2 = StableDiffusionXLImg2ImgPipeline.from_single_file('/kaggle/working/animagine-xl-4.0.safetensors', torch_dtype=torch.float16).to('cuda')
pipe2.load_lora_weights(lora_dir, weight_name='shiori_oneesan_v4.safetensors')
for name, prompt, img in bases:
    big = img.resize((1536, 1536), Image.LANCZOS)
    g = torch.Generator(device='cuda').manual_seed(42)
    refined = pipe2(prompt, negative_prompt=neg, image=big, strength=0.35, num_inference_steps=30, guidance_scale=7, generator=g).images[0]
    refined.save('/kaggle/working/output/' + name)
    print('saved', name)
bases[0][2]
